In [1]:

import pandas as pd
import numpy as np
import os
import json
import time
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Import from our project
from src.embeddings import (
    pca_embedding, lle_embedding, umap_embedding, 
    wavelet_embedding, fft_embedding, AE_embedding,
    graph_embedding, tda_embedding, 
    nnclr_cnn_embedding, nnclr_lstm_embedding
)
from src.classifiers import (
    optimize_LOGRG, optimize_DT, optimize_RF, optimize_KNN, 
    optimize_XGBOOST, optimize_SVM, optimize_MLP, optimize_NB
)
from src.utils.preprocessing import load_dataset, prepare_dataset
from src.utils.evaluation import (
    evaluate_embeddings, visualize_embeddings, 
    compare_classifiers, generate_classification_report
)
from src.utils.visualization import umap_plt, plot_embedding_comparison



I0000 00:00:1792412592.015229  808586 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
import sys
sys.path.insert(0, "src")
from run_experiments import *

In [3]:
from sklearn.model_selection import train_test_split
# Get the list of countries and their corresponding continents from the provided URL.
url_data = (r'https://raw.githubusercontent.com/oderofrancis/rona/main/Countries-Continents.csv')
data_countries = pd.read_csv(url_data)


""" 
List of countries in the ntl data that are not present in the provided countries-continents CSV file.
{'DR Congo',
 'Mayotte',
 'Myanmar',
 'Myanmar (post-coup)',
 'México',
 'Palestine (Israel)',
 'Puerto Rico',
 'Réunion',
 'Taiwan',
 'United States'}
"""
# Do I make MENA a seperate region? I think I should.

missing_countries = {
    "Continent": [
        "Africa",
        "Africa",
        "Asia",
        "Asia",
        "North America",
        "Asia",
        "North America",
        "Africa",
        "Asia",
        "North America",
    ],
    "Country": [
        "DR Congo",
        "Mayotte",
        "Myanmar",
        "Myanmar (post-coup)",
        "México",
        "Palestine (Israel)",
        "Puerto Rico",
        "Réunion",
        "Taiwan",
        "United States",
    ],
}

data_countries = pd.concat([
    data_countries,
    pd.DataFrame(missing_countries)
], ignore_index=True)
data_countries.rename(columns={"Country": "country"}, inplace=True)


ntl_data = pd.read_csv('final_pre365/ntl_cases.csv')[['case_id', 'time', 'ntl_filled']]
ntl_data['time'] = pd.to_datetime(ntl_data['time'])
ntl_data = ntl_data.sort_values(['case_id', 'time']).reset_index(drop=True)
ntl_data['time_step'] = ntl_data.groupby('case_id').cumcount()

cases = pd.read_csv('final_pre365/cases.csv')
cases['event_date'] = pd.to_datetime(cases['event_date'])
"""
# Split data into train (test) and val
split_cases = pd.read_csv('final/split.csv')
cases = cases.merge(
    split_cases[['case_id', 'split']],
    on='case_id',
    how='left'
)
"""
"""
train_idx, val_idx = train_test_split(
    cases.index,
    test_size=0.2,
    stratify=cases['event_type'],
    random_state=42,
)

cases['split'] = 'train'
cases.loc[val_idx, 'split'] = 'val'

ntl_data = ntl_data.merge(
    cases[['case_id', 'split']],
    on='case_id',
    how='left'
)
"""
# Add the continent/

cases = cases.merge(
    data_countries[['country', 'Continent']],
    on='country',
    how='left'
).set_index('case_id')

event_types = sorted(cases['event_type'].unique())

for date_col in ['event_start', 'event_end', 'window_start', 'window_end']:
    cases[date_col] = pd.to_datetime(cases[date_col] )

feature_list = ['event_type', 'Continent']
feature_dict = dict()
for feature in feature_list:
    feature_value_dict = {feature_value : i for i, feature_value in enumerate(cases[feature].unique())}
    
    cases[feature + '_class'] = cases[feature].map(feature_value_dict)
    feature_dict[feature] = feature_value_dict
    

In [4]:
colors = {'Continent': {
            'Africa':'green', 
            'Asia': 'red',
            'Europe': 'darkviolet',
            'North America': 'blue',
            'South America': 'darkorange',
            'Oceania': 'mediumturquoise'}, 
            'event_type': {'conflict': 'tomato', 'cyclone': 'cadetblue', 'earthquake': 'mediumseagreen', 'flood': 'dodgerblue'}}

In [5]:
from sklearn.preprocessing import StandardScaler
def calculate_window(ntl_data, cases, d_window):
    case_ids = dict()
    frames = []                       # collect pieces, concat once at the end
    cases_window = cases.copy()

    # plain Python float -> no NumPy "generic unit" timedelta warning
    window = pd.Timedelta(days=float(d_window))

    def _rescale(frame):
        frame = frame.copy()          # avoids SettingWithCopyWarning
        frame['time_step'] = frame.groupby('case_id').cumcount()
        frame['ntl_std'] = StandardScaler().fit_transform(
            frame['ntl_filled'].values.reshape(-1, 1)
        ).ravel()
        return frame

    for case_id, df in ntl_data.groupby('case_id'):
        if cases.loc[case_id, 'n_events'] == 1:
            event_date = cases.loc[case_id, 'event_date']
            start_date = event_date - window
            end_date = event_date + window

            cases_window.loc[case_id, 'window_start'] = start_date
            cases_window.loc[case_id, 'window_end'] = end_date

            df = df[(df['time'] >= start_date) & (df['time'] <= end_date)]
            case_ids[case_id] = case_id

            if not df.empty and not df.isna().all().all():
                frames.append(_rescale(df))

        else:
            event_dates = pd.to_datetime(
                list(set(cases.loc[case_id, 'event_dates'].split(';')))
            )
            cases_window.drop(index=case_id, inplace=True)

            for i, event_date in enumerate(event_dates):
                new_case_id = f"{case_id}_event_{i+1}"
                start_date = event_date - window
                end_date = event_date + window

                cases_window.loc[new_case_id] = cases.loc[case_id]
                cases_window.loc[new_case_id, 'event_date'] = event_date
                cases_window.loc[new_case_id, 'window_start'] = start_date
                cases_window.loc[new_case_id, 'window_end'] = end_date

                case_ids[new_case_id] = case_id

                df_event = df[(df['time'] >= start_date) & (df['time'] <= end_date)]

                if not df_event.empty and not df_event.isna().all().all():
                    df_event = df_event.copy()
                    df_event['case_id'] = new_case_id   # set before cumcount
                    frames.append(_rescale(df_event))

    new_ntl = (pd.concat(frames, ignore_index=True)
               if frames else pd.DataFrame(columns=ntl_data.columns))
    
    cases_window['original_id'] = cases_window.index.map(case_ids)

    return case_ids, new_ntl, cases_window

In [6]:
#d_window = 30

#case_ids,  ntl_window, cases_window = calculate_window(ntl_data, cases, d_window)

In [7]:
#results = {
#        'dataset': d_window,
 #       'embedding_results': {},
  #      'classifier_results': {},
   #     'combined_results': {}
    #7}

In [8]:
cases

,event_type,source_cluster_id,city,country,event_date,event_dates,n_events,catalog_event_date,catalog_event_dates,shift_reason,event_start,event_end,window_anchor,window_start,window_end,notes,raster_path,Continent,event_type_class,Continent_class
case_id,,,,,,,,,,,,,,,,,,,,
earthquake_2,earthquake,2,Sagaing,Myanmar,2025-03-28,2025-03-28,1,2025-03-28 06:20:52.715,2025-03-28,NaN,2025-03-28 06:20:52.715,2025-03-28 06:20:52.715,2025-03-28,2024-03-28,2025-09-24,NaN,rasters/earthquake_2.npz,Asia,0,0
earthquake_3,earthquake,3,Iquique,Chile,2014-04-03,2014-04-03,1,2014-04-03 02:43:13.110,2014-04-03;2014-04-03,NaN,2014-04-03 01:58:30.530,2014-04-03 02:43:13.110,2014-04-03,2013-04-03,2014-09-30,NaN,rasters/earthquake_3.npz,South America,0,1
earthquake_7,earthquake,7,Catia La Mar [La Guaira],Venezuela,2026-06-24,2026-06-24,1,2026-06-24 22:05:04.012,2026-06-24,NaN,2026-06-24 22:05:04.012,2026-06-24 22:05:04.012,2026-06-24,2025-06-24,2026-10-08,NaN,rasters/earthquake_7.npz,South America,0,1
earthquake_11,earthquake,11,Halabja,Iraq,2017-11-20,2017-11-20,1,2017-11-12 18:18:17.180,2017-11-12,NaN,2017-11-12 18:18:17.180,2017-11-12 18:18:17.180,2017-11-12,2016-11-12,2018-05-11,NaN,rasters/earthquake_11.npz,Asia,0,0
earthquake_29,earthquake,29,Valparaíso,Chile,2017-05-05,2017-05-05,1,2017-04-24 21:38:30.820,2017-04-23;2017-04-24,NaN,2017-04-23 02:36:07.830,2017-04-24 21:38:30.820,2017-04-23,2016-04-23,2017-10-21,NaN,rasters/earthquake_29.npz,South America,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
conflict_615_0,conflict,615,Mek'elē,Ethiopia,2020-12-01,2020-12-01;2021-02-16;2021-04-01;2021-06-30;20...,6,NaN,NaN,NaN,2020-12-01 00:00:00.000,2021-12-01 00:00:00.000,2020-12-01,2019-12-02,2022-06-01,NaN,rasters/conflict_615_0.npz,Africa,3,4
conflict_616_0,conflict,616,Quiha,Ethiopia,2020-12-03,2020-12-03,1,NaN,NaN,NaN,2020-12-03 00:00:00.000,2020-12-03 00:00:00.000,2020-12-03,2019-12-04,2021-06-03,NaN,rasters/conflict_616_0.npz,Africa,3,4
conflict_617_0,conflict,617,Wukro,Ethiopia,2020-11-16,2020-11-16;2021-02-18;2021-04-01;2021-06-29,4,NaN,NaN,NaN,2020-11-16 00:00:00.000,2021-06-29 00:00:00.000,2020-11-16,2019-11-17,2021-12-29,NaN,rasters/conflict_617_0.npz,Africa,3,4


In [11]:
ntl_data

,case_id,time,ntl_filled,time_step,split_x,split_y
0,conflict_100_0,2013-11-21,26.959764,0,train,train
1,conflict_100_0,2013-11-22,16.383764,1,train,train
2,conflict_100_0,2013-11-23,23.448994,2,train,train
3,conflict_100_0,2013-11-24,24.762720,3,train,train
4,conflict_100_0,2013-11-25,23.340550,4,train,train
...,...,...,...,...,...,...
172095,flood_942,2024-06-27,6.411082,557,train,train
172096,flood_942,2024-06-28,6.411082,558,train,train
172097,flood_942,2024-06-29,6.411082,559,train,train
172098,flood_942,2024-06-30,6.411082,560,train,train


In [ ]:
embedding_methods = ['pca', 'lle', 'umap']#,'nnclr_cnn']
#embedding_methods = ['nnclr_cnn']
classifiers = ['logreg', 'dt', 'rf', 'knn','xgboost', 'svm', 'mlp','nb']
for feature in feature_list:
    train_idx, val_idx = train_test_split(
        cases.index,
        test_size=0.2,
        stratify=cases[feature],
        random_state=42,
    )

    cases[f'split_{feature}'] = 'train'
    cases.loc[val_idx, f'split_{feature}'] = 'val'

    ntl_data = ntl_data.merge(
    cases[[f'split_{feature}']].reset_index(inplace=False),
    on='case_id',
    how='left')
            
    
    for d_window in [7, 14, 30]:
        case_ids,  ntl_window, cases_window = calculate_window(ntl_data, cases, d_window)
        results = {
                'dataset': d_window,
                'embedding_results': {},
                'classifier_results': {},
                'combined_results': {}
            }
        X_train_norm = ntl_window[ntl_window[f'split_{feature}'] == 'train'].pivot(index='case_id', columns='time_step', values='ntl_std').dropna()
        X_val_norm = ntl_window[ntl_window[f'split_{feature}'] == 'val'].pivot(index='case_id', columns='time_step', values='ntl_std').dropna()
        X_test_norm = X_val_norm
        #X_full = ntl_window.pivot(index='case_id', columns='time_step', values='ntl_std').dropna()
        
        

        dataset_path = f'results/{feature}/data_{str(d_window)}/data_{str(d_window)}'
        os.makedirs(f'results/{feature}/data_{str(d_window)}',exist_ok=True)

        output_dir = f'plots/{feature}/data_{str(d_window)}'
        os.makedirs(output_dir ,exist_ok=True)

        y_train = cases_window.loc[X_train_norm.index, feature + '_class'].values
        y_val = cases_window.loc[X_val_norm.index, feature + '_class'].values
        y_test =y_val
        
        y_train_labels = cases_window.loc[X_train_norm.index, feature].values
        y_val_labels = cases_window.loc[X_val_norm.index, feature ].values
        y_test_labels =y_val_labels

        for embed_method in embedding_methods:
            print(f"\nEvaluating embedding method: {embed_method}")
            # Run embedding
            try:
                train_emb, val_emb, test_emb, embed_time = run_embedding_experiment(
            X_train_norm.values, X_val_norm.values, X_test_norm, y_train, y_val, y_test, embed_method
                )

                # Store embedding results
                results['embedding_results'][embed_method] = {
                'time': embed_time,
                'shape': train_emb.shape if hasattr(train_emb, 'shape') else None
                }

                # Visualize embeddings
                try:
                    if embed_method in ['nnclr_cnn', 'nnclr_lstm']:
                        # For NNCLR methods, use umap_plt from visualization module
                        umap_plt(
                        train_emb, val_emb, test_emb, 
                        y_train, y_val, y_test, 
                        f"{embed_method.upper()} Embedding (window = {d_window})"
                            )
                    else:
                        # For other methods, use visualize_embeddings from evaluation module
                        fig = visualize_embeddings(
                        train_emb,labels= y_train_labels, method='tsne', colors= colors[feature],
                        title=f'{embed_method.upper()} Embedding Visualization (window = {d_window})'
                            )
                        fig.savefig(os.path.join(output_dir, f'{embed_method}_visualization.png'))
                        plt.close(fig)
                except Exception as viz_error:
                    print(f"  - Warning: Visualization failed: {viz_error}")

                # Run classifier experiments
                for clf_name in classifiers:
                    print(f"  - Testing classifier: {clf_name}")

                    experiment_name = f"{dataset_path}_{embed_method}_{clf_name}"

                    try:
                        best_params, test_score, clf_time = run_classifier_experiment(
                train_emb, val_emb, test_emb, y_train, y_val, y_test, 
                clf_name, experiment_name
                    )

                        # Store results
                        if embed_method not in results['combined_results']:
                            results['combined_results'][embed_method] = {}

                        results['combined_results'][embed_method][clf_name] = {
                    'accuracy': test_score,
                    'params': best_params,
                    'time': clf_time
                        }

                        print(f"    Accuracy: {test_score:.4f}, Time: {clf_time:.2f}s")

                    except Exception as e:
                        print(f"    Error with classifier {clf_name}: {e}")
                        results['combined_results'][embed_method][clf_name] = {
                    'error': str(e)
                    }
                    

            except Exception as e:
                print(f"Error with embedding method {embed_method}: {e}")
                results['embedding_results'][embed_method] = {
                'error': str(e)
                    }
        # Save results
        rows = []

        for embedding, classifiers in results["combined_results"].items():
            for classifier, metrics in classifiers.items():
                rows.append({
                    "embedding": embedding,
                    "classifier": classifier,
                    "accuracy": metrics["accuracy"],
                    "time": metrics["time"],
                    "params": metrics["params"]
                })

        results_df = pd.DataFrame(rows)    
        results_df.to_csv(dataset_path+'_results.csv')


Evaluating embedding method: pca


[I 2026-10-19 14:46:07,920] A new study created in memory with name: no-name-c60faa63-4870-4ce3-ac15-d7ed96c17f58
[I 2026-10-19 14:46:07,935] Trial 0 finished with value: 0.4675324675324675 and parameters: {'C': 0.018281667216737203, 'fit_intercept': False, 'solver': 'saga', 'penalty': 'elasticnet', 'l1_ratio': 0.6045572732356488}. Best is trial 0 with value: 0.4675324675324675.
[I 2026-10-19 14:46:07,951] Trial 1 finished with value: 0.5064935064935064 and parameters: {'C': 0.1356046262445642, 'fit_intercept': True, 'solver': 'saga', 'penalty': 'elasticnet', 'l1_ratio': 0.3940811665404621}. Best is trial 1 with value: 0.5064935064935064.
[I 2026-10-19 14:46:07,966] Trial 2 finished with value: 0.36363636363636365 and parameters: {'C': 0.0761416973744006, 'fit_intercept': False, 'solver': 'saga', 'penalty': 'elasticnet', 'l1_ratio': 0.13671056011179405}. Best is trial 1 with value: 0.5064935064935064.
[I 2026-10-19 14:46:07,981] Trial 3 finished with value: 0.4675324675324675 and param

  - Testing classifier: logreg


[I 2026-10-19 14:46:08,397] Trial 5 finished with value: 0.44155844155844154 and parameters: {'C': 89.88014098122673, 'fit_intercept': True, 'solver': 'saga', 'penalty': 'elasticnet', 'l1_ratio': 0.06748734675628343}. Best is trial 1 with value: 0.5064935064935064.
[I 2026-10-19 14:46:08,415] Trial 6 finished with value: 0.33766233766233766 and parameters: {'C': 114.89465081823104, 'fit_intercept': False, 'solver': 'saga', 'penalty': 'elasticnet', 'l1_ratio': 0.04420134126284192}. Best is trial 1 with value: 0.5064935064935064.
[I 2026-10-19 14:46:08,432] Trial 7 finished with value: 0.4675324675324675 and parameters: {'C': 1.1904622883210919e-09, 'fit_intercept': True, 'solver': 'saga', 'penalty': 'elasticnet', 'l1_ratio': 0.9784651031255993}. Best is trial 1 with value: 0.5064935064935064.
[I 2026-10-19 14:46:08,450] Trial 8 finished with value: 0.4675324675324675 and parameters: {'C': 3.6854593156891155e-06, 'fit_intercept': False, 'solver': 'saga', 'penalty': 'elasticnet', 'l1_rati

0.014403343200683594  seconds
Classification report saved as results/Continent/data_7/data_7_pca_logreg_classification_report.txt
    Accuracy: 0.5195, Time: 6.82s
  - Testing classifier: dt


...

0.0035583972930908203  seconds
Classification report saved as results/Continent/data_30/data_30_umap_svm_classification_report.txt
    Accuracy: 0.4805, Time: 0.53s
  - Testing classifier: mlp


/home/zeineb/.conda/envs/timeseries_py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
[I 2026-10-19 15:16:16,344] Trial 1 finished with value: 0.44155844155844154 and parameters: {'n_layers': 3, 'n_neurons_layer0': 7, 'n_neurons_layer1': 29, 'n_neurons_layer2': 31, 'activation': 'identity', 'solver': 'lbfgs', 'alpha': 0.0467946163269153}. Best is trial 0 with value: 0.4675324675324675.
/home/zeineb/time-series-embedding/src/classifiers/neural_models.py:29: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(...,

{'activation': 'tanh', 'solver': 'lbfgs', 'alpha': 3.482923944218014e-05, 'hidden_layer_sizes': [7, 73, 54]}
Classification report saved as results/Continent/data_30/data_30_umap_mlp_classification_report_2026-10-19_15-24-23.txt
    Accuracy: 0.4675, Time: 487.19s
  - Testing classifier: nb


[I 2026-10-19 15:24:23,330] Trial 72 finished with value: 0.4155844155844156 and parameters: {'var_smoothing': 0.002418618389230005}. Best is trial 4 with value: 0.4155844155844156.
[I 2026-10-19 15:24:23,333] Trial 73 finished with value: 0.4025974025974026 and parameters: {'var_smoothing': 0.000576123764705626}. Best is trial 4 with value: 0.4155844155844156.
[I 2026-10-19 15:24:23,336] Trial 74 finished with value: 0.4155844155844156 and parameters: {'var_smoothing': 0.0050680139764463904}. Best is trial 4 with value: 0.4155844155844156.
[I 2026-10-19 15:24:23,339] Trial 75 finished with value: 0.4155844155844156 and parameters: {'var_smoothing': 0.001623604832098941}. Best is trial 4 with value: 0.4155844155844156.
[I 2026-10-19 15:24:23,341] Trial 76 finished with value: 0.4025974025974026 and parameters: {'var_smoothing': 0.0009056028918615882}. Best is trial 4 with value: 0.4155844155844156.
[I 2026-10-19 15:24:23,344] Trial 77 finished with value: 0.4155844155844156 and paramet

Best hyperparameters:  {'var_smoothing': 0.0016395896164580785}
Best validation accuracy:  0.4155844155844156
Test accuracy: 0.4155844155844156
Time taken: 0.0012116432189941406 seconds
Classification report saved as results/Continent/data_30/data_30_umap_nb_classification_report_2026-10-19_15-24-23.txt
    Accuracy: 0.4156, Time: 0.25s


In [ ]:
"""
embedding_methods = ['pca', 'lle', 'umap','nnclr_cnn']
#embedding_methods = ['nnclr_cnn']
classifiers = ['logreg', 'dt', 'rf', 'knn','xgboost', 'svm', 'mlp','nb']
X_train_norm = ntl_window[ntl_window['split'] == 'train'].pivot(index='case_id', columns='time_step', values='ntl_std').dropna()
X_val_norm = ntl_window[ntl_window['split'] == 'val'].pivot(index='case_id', columns='time_step', values='ntl_std').dropna()
X_test_norm = X_val_norm
X_full = ntl_window.pivot(index='case_id', columns='time_step', values='ntl_std').dropna()

dataset_path = f'results/data_{str(d_window)}/data_{str(d_window)}'
os.makedirs(f'results/data_{str(d_window)}',exist_ok=True)

output_dir = f'plots/data_{str(d_window)}'
os.makedirs(output_dir ,exist_ok=True)

y_train = cases_window.loc[X_train_norm.index, 'event_type'].values
y_val = cases_window.loc[X_val_norm.index, 'event_type'].values
y_test =y_val

for embed_method in embedding_methods:
    print(f"\nEvaluating embedding method: {embed_method}")
    # Run embedding
    try:
        train_emb, val_emb, test_emb, embed_time = run_embedding_experiment(
     X_train_norm.values, X_val_norm.values, X_test_norm, y_train, y_val, y_test, embed_method
        )

        # Store embedding results
        results['embedding_results'][embed_method] = {
        'time': embed_time,
        'shape': train_emb.shape if hasattr(train_emb, 'shape') else None
        }

        # Visualize embeddings
        try:
            if embed_method in ['nnclr_cnn', 'nnclr_lstm']:
                # For NNCLR methods, use umap_plt from visualization module
                umap_plt(
                train_emb, val_emb, test_emb, 
                y_train, y_val, y_test, 
                f"{embed_method.upper()} Embedding"
                    )
            else:
                # For other methods, use visualize_embeddings from evaluation module
                fig = visualize_embeddings(
                train_emb, y_train, method='tsne', 
                title=f'{embed_method.upper()} Embedding Visualization'
                    )
                fig.savefig(os.path.join(output_dir, f'{embed_method}_visualization.png'))
                plt.close(fig)
        except Exception as viz_error:
            print(f"  - Warning: Visualization failed: {viz_error}")

        # Run classifier experiments
        for clf_name in classifiers:
            print(f"  - Testing classifier: {clf_name}")

            experiment_name = f"{dataset_path}_{embed_method}_{clf_name}"

            try:
                best_params, test_score, clf_time = run_classifier_experiment(
        train_emb, val_emb, test_emb, y_train, y_val, y_test, 
        clf_name, experiment_name
            )

                # Store results
                if embed_method not in results['combined_results']:
                    results['combined_results'][embed_method] = {}

                results['combined_results'][embed_method][clf_name] = {
            'accuracy': test_score,
            'params': best_params,
            'time': clf_time
                }

                print(f"    Accuracy: {test_score:.4f}, Time: {clf_time:.2f}s")

            except Exception as e:
                print(f"    Error with classifier {clf_name}: {e}")
                results['combined_results'][embed_method][clf_name] = {
            'error': str(e)
            }

    except Exception as e:
        print(f"Error with embedding method {embed_method}: {e}")
        results['embedding_results'][embed_method] = {
        'error': str(e)
            }
            
            """

'\nembedding_methods = [\'pca\', \'lle\', \'umap\',\'nnclr_cnn\']\n#embedding_methods = [\'nnclr_cnn\']\nclassifiers = [\'logreg\', \'dt\', \'rf\', \'knn\',\'xgboost\', \'svm\', \'mlp\',\'nb\']\nX_train_norm = ntl_window[ntl_window[\'split\'] == \'train\'].pivot(index=\'case_id\', columns=\'time_step\', values=\'ntl_std\').dropna()\nX_val_norm = ntl_window[ntl_window[\'split\'] == \'val\'].pivot(index=\'case_id\', columns=\'time_step\', values=\'ntl_std\').dropna()\nX_test_norm = X_val_norm\nX_full = ntl_window.pivot(index=\'case_id\', columns=\'time_step\', values=\'ntl_std\').dropna()\n\ndataset_path = f\'results/data_{str(d_window)}/data_{str(d_window)}\'\nos.makedirs(f\'results/data_{str(d_window)}\',exist_ok=True)\n\noutput_dir = f\'plots/data_{str(d_window)}\'\nos.makedirs(output_dir ,exist_ok=True)\n\ny_train = cases_window.loc[X_train_norm.index, \'event_type\'].values\ny_val = cases_window.loc[X_val_norm.index, \'event_type\'].values\ny_test =y_val\n\nfor embed_method in embed

In [10]:
cases

,event_type,source_cluster_id,city,country,event_date,event_dates,n_events,catalog_event_date,catalog_event_dates,shift_reason,...,event_end,window_anchor,window_start,window_end,notes,raster_path,Continent,event_type_class,Continent_class,split
case_id,,,,,,,,,,,,,,,,,,,,,
earthquake_2,earthquake,2,Sagaing,Myanmar,2025-03-28,2025-03-28,1,2025-03-28 06:20:52.715,2025-03-28,NaN,...,2025-03-28 06:20:52.715,2025-03-28,2024-03-28,2025-09-24,NaN,rasters/earthquake_2.npz,Asia,0,0,train
earthquake_3,earthquake,3,Iquique,Chile,2014-04-03,2014-04-03,1,2014-04-03 02:43:13.110,2014-04-03;2014-04-03,NaN,...,2014-04-03 02:43:13.110,2014-04-03,2013-04-03,2014-09-30,NaN,rasters/earthquake_3.npz,South America,0,1,val
earthquake_7,earthquake,7,Catia La Mar [La Guaira],Venezuela,2026-06-24,2026-06-24,1,2026-06-24 22:05:04.012,2026-06-24,NaN,...,2026-06-24 22:05:04.012,2026-06-24,2025-06-24,2026-10-08,NaN,rasters/earthquake_7.npz,South America,0,1,train
earthquake_11,earthquake,11,Halabja,Iraq,2017-11-20,2017-11-20,1,2017-11-12 18:18:17.180,2017-11-12,NaN,...,2017-11-12 18:18:17.180,2017-11-12,2016-11-12,2018-05-11,NaN,rasters/earthquake_11.npz,Asia,0,0,val
earthquake_29,earthquake,29,Valparaíso,Chile,2017-05-05,2017-05-05,1,2017-04-24 21:38:30.820,2017-04-23;2017-04-24,NaN,...,2017-04-24 21:38:30.820,2017-04-23,2016-04-23,2017-10-21,NaN,rasters/earthquake_29.npz,South America,0,1,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
conflict_615_0,conflict,615,Mek'elē,Ethiopia,2020-12-01,2020-12-01;2021-02-16;2021-04-01;2021-06-30;20...,6,NaN,NaN,NaN,...,2021-12-01 00:00:00.000,2020-12-01,2019-12-02,2022-06-01,NaN,rasters/conflict_615_0.npz,Africa,3,4,train
conflict_616_0,conflict,616,Quiha,Ethiopia,2020-12-03,2020-12-03,1,NaN,NaN,NaN,...,2020-12-03 00:00:00.000,2020-12-03,2019-12-04,2021-06-03,NaN,rasters/conflict_616_0.npz,Africa,3,4,train
conflict_617_0,conflict,617,Wukro,Ethiopia,2020-11-16,2020-11-16;2021-02-18;2021-04-01;2021-06-29,4,NaN,NaN,NaN,...,2021-06-29 00:00:00.000,2020-11-16,2019-11-17,2021-12-29,NaN,rasters/conflict_617_0.npz,Africa,3,4,train
